In [42]:
import os
import re
import datetime as dt
from dotenv import load_dotenv
from pathlib import Path

import msal
import pandas as pd
import requests
import numpy as np
import json

In [35]:
schema = {
    "PlanID": "string",
    "BucketID": "string",
    "TaskId": "string",
    "AssignedUserId": "string",
    "CompletedByUserId": "string",
    "TaskTitle": "string",
    "Description": "string",
    "TaskOrderHint": "string",
    "TaskAssignee": "string",
    "TaskAssigneeEmail": "string",
    "completedBy": "string",
    "PreviewType": "string",
    "appliedCategories": "string",
    "TaskAssignments": "string",
    "TaskChecklistItemCount": "Int64",
    "TaskActiveChecklistItemCount": "Int64",
    "TaskPriority": "Int64",
    "assigneePriority": "string",
    "TaskHasDescription": "string",
    "TaskCompleted": "string",
    "TaskStartDateTime": "datetime64[ns]",
    "TaskCreatedDateTime": "datetime64[ns]",
    "TaskDueDateTime": "datetime64[ns]",
    "TaskCompletedDateTime": "datetime64[ns]",
}

date_cols = [k for k, v in schema.items() if v.startswith("datetime")]
dtype_cols = {k: v for k, v in schema.items() if k not in date_cols}


In [36]:
df_events_staging = pd.read_csv(
    r"C:\Users\criss\TP Caterers\TCP BI - Documents\Data\planner_data_pipeline\data\staging\Fact_Tasks.csv",
    dtype=dtype_cols,
    parse_dates=date_cols,
    keep_default_na=True
)

df_events_staging.shape

(16088, 25)

In [37]:
df_events_staging.dtypes

PlanID                          string[python]
BucketID                        string[python]
TaskTitle                       string[python]
TaskOrderHint                   string[python]
assigneePriority                string[python]
TaskPercentComplete                      int64
TaskStartDateTime               datetime64[ns]
TaskCreatedDateTime             datetime64[ns]
TaskDueDateTime                 datetime64[ns]
TaskHasDescription              string[python]
PreviewType                     string[python]
TaskCompletedDateTime           datetime64[ns]
TaskChecklistItemCount                   Int64
TaskActiveChecklistItemCount             Int64
TaskPriority                             Int64
TaskId                          string[python]
completedBy                     string[python]
appliedCategories               string[python]
TaskAssignments                 string[python]
CompletedByUserId               string[python]
TaskCompleted                   string[python]
Description  

In [38]:
df_buckets_staging = pd.read_csv(r"C:\Users\criss\TP Caterers\TCP BI - Documents\Data\planner_data_pipeline\data\staging\Dim_Buckets.csv", 
                                 dtype={'BucketID': 'string', 
                                        'BucketKey': 'Int64'})
df_events_staging.shape

(16088, 25)

In [33]:
df_buckets_staging.head()

,BucketID,BucketKey
0,Pi6hZOYjz0W-57WmivzXuWUALmxC,1
1,3cBNzZ-Dfke6H9H8P7AwJ2UAP8Cf,1
2,tH4p0zt0W0OZPSUT1DNX4mUAHGUW,1
3,j5cmR9lg50KOSYjvWEC4M2UAGBmf,2
4,t3hQTrgzskGZhVkhA5vMnWUAM65R,3


In [39]:
df_events_prod = df_events_staging.merge(df_buckets_staging, on= 'BucketID', how='left')


In [40]:
df_events_prod.dtypes

PlanID                          string[python]
BucketID                        string[python]
TaskTitle                       string[python]
TaskOrderHint                   string[python]
assigneePriority                string[python]
TaskPercentComplete                      int64
TaskStartDateTime               datetime64[ns]
TaskCreatedDateTime             datetime64[ns]
TaskDueDateTime                 datetime64[ns]
TaskHasDescription              string[python]
PreviewType                     string[python]
TaskCompletedDateTime           datetime64[ns]
TaskChecklistItemCount                   Int64
TaskActiveChecklistItemCount             Int64
TaskPriority                             Int64
TaskId                          string[python]
completedBy                     string[python]
appliedCategories               string[python]
TaskAssignments                 string[python]
CompletedByUserId               string[python]
TaskCompleted                   string[python]
Description  

In [ ]:
def get_user(user_id):
    url = f"{GRAPH}/users/{user_id}"
    params = {
        "$select": "id,displayName,mail,userPrincipalName,jobTitle,department,accountEnabled"
    }
    return graph_get(url, headers=headers, params=params)
task_assignee_userid = df_tasks['AssignedUserId'].dropna().unique()

In [ ]:
user_ids = df_events_prod['CompletedByUserId'].dropna().unique()

dim_user_rows = []

for uid in user_ids:
    try:
        u = get_user(uid)
        dim_user_rows.append(u)
    except requests.HTTPError as e:
        # Common reasons: user deleted, guest, no permission, or id isn't a user
        print("Failed user:", uid, str(e))
        dim_user_rows.append({"id": uid})  # keep id so joins still work

DimUser = pd.DataFrame(dim_user_rows).rename(columns={"id": "UserID"})
print("DimUser rows:", len(DimUser))
# DimUser.head()

DimUser = DimUser[['UserID', 'displayName', 'userPrincipalName']]
# DimUser.to_csv("../02 Data/Dim_User.csv", index=False)

DimUser.rename(columns={
    "displayName":"TaskCompletedByName", 
    "userPrincipalName": "TaskCompletedByUserEmail"
    }, inplace=True)

df_tasks_out = df_tasks_out.merge(DimUser, left_on='CompletedByUserId', right_on='UserID', how='left')

In [41]:
df_events_prod[[
'TaskId'
,'TaskTitle'
,'TaskDescription'
,'TaskHasDescription'
,'TaskOrderHint'
,'TaskAssignee'
,'TaskAssigneeEmail'
,'TaskDueDateTime'
,'TaskStartDateTime'
,'TaskCompletedDate'
,'TaskCompleted'
,'TaskCompletedByName'
,'TaskCompletedByUserEmail'
,'TaskPriority'
,'TaskPreviewType'
,'TaskChecklistItemCount'
,'TaskActiveChecklistItemCount'
,'Label'
,'BucketKey'
,'PlanID'
,'LastRun'
]]

KeyError: "['TaskDescription', 'TaskCompletedDate', 'TaskCompletedByName', 'TaskCompletedByUserEmail', 'TaskPreviewType', 'Label', 'LastRun'] not in index"